# Group Theory with Z3

In this notebook we show how to investigate topics in [group theory](https://en.wikipedia.org/wiki/Group_theory) using the constraint solver `Z3`.  In particular, 
we want to show that not every group is *commutative* by constructing a group in which the commutative law does not hold.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

A [group](https://en.wikipedia.org/wiki/Group_(mathematics)) is defined as a triple 
$$ \mathcal{G} = \langle G, \mathrm{e}, \circ \rangle $$
such that
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary operation on $G$ such that the following axioms are satisfied:
  * $\forall x: \mathrm{e} \circ  x = x$,
  * $\forall x: \exists{y}: y \circ x = \mathrm{e}$,
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$.

The group $\mathcal{G}$ is *commutative* if, additionally, the following holds:
  $$\forall x: \forall y: x \circ y = y \circ x. $$
For example, the triple $\langle \mathbb{Z}, 0, + \rangle$ is a commutative group. 
It is natural to ask whether every group is commutative, i.e. whether the formula 
$\forall x: \forall y: x \circ y = y \circ x$ is a logical consequence of the three group axioms.
To this end we can try to prove the following
$$ \{ \forall x: \mathrm{e} \circ  x = x,\; \forall x: \exists{y}: y \circ x = \mathrm{e},\;
      \forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z) \} \models \forall x: \forall y: x \circ y = y \circ x
$$
using automatic theorem proving.  We use the theorem prover *Vampire*.  The function `shell` is defined in `setup.fsx`.  It executes a shell command and prints its output.

In [3]:
shell "cat ../../Vampire/non-commutative-group.tptp"

% We try to prove that every group is commutative.

% e is the neutral element.
fof(a1, axiom, ! [X] : mult(e,X) = X).

% For every element X there is a left inverse element Y.
fof(a2, axiom, ! [X] : ? [Y]: mult(Y, X) = e).

% The law of associativity holds.
fof(a3, axiom, ! [X,Y,Z] : mult(mult(X, Y), Z) = mult(X, mult(Y, Z))).

% Question: Does the law of commutativity hold?
fof(commute, conjecture, ! [X, Y] : mult(X, Y) = mult(Y, X)).


In [4]:
shell "vampire --input_syntax tptp ../../Vampire/non-commutative-group.tptp"

% Time limit reached!
% ------------------------------
% Version: Vampire 4.7 (commit )
% Termination reason: Time limit
% Termination phase: Saturation

% Memory used [KB]: 2564606
% Time elapsed: 60.100 s
% ------------------------------
% ------------------------------


However, as this attempt produces no result we can instead try to refute the hypothesis that every group is commutative by constructing a group that is not commutative.  We will try to find a *finite* non-commutative group.
In a finite group with $n$ elements we can assume without loss of generality that the elements
are the natural numbers $\{ 0, 1, 2, \cdots, n-1 \}$.  Furthermore, by reordering the elements 
we can assume that $0$ is the neutral element $\mathrm{e}$, i.e. we have $\mathrm{e} = 0$.
Therefore, in order to construct a group we only have to find a binary operation $\circ$ on the set
$\{ 0, \cdots, n-1 \}$ that satisfies the three axioms specifying a group.  We will specify this operation 
as a map `Variables` that maps pairs of the form $\langle x, y \rangle \in \{ 0, \cdots, n-1 \} \times \{ 0, \cdots, n-1 \}$
to `Z3` variables.  The value of the variable `Variables[(x, y)]` is $x \circ y$.

In [5]:
let n = 3
let Variables = Map [ for row in 0 .. n - 1 do
                        for col in 0 .. n - 1 ->
                            ((row, col), Int (sprintf "V%d%d" row col)) ]
Variables

map [((0, 0), V00); ((0, 1), V01); ((0, 2), V02); ((1, 0), V10); ((1, 1), V11); ((1, 2), V12); ((2, 0), V20); ((2, 1), V21); ((2, 2), V22)]

The idea is that we have
$$ x \circ y = z \;\Longleftrightarrow\; \texttt{Variables}[x, y] = z.$$

The axiom 
$$ \forall x: \mathrm{e} \circ x = x $$
that $0$ is the left neutral element of the binary operation $\circ$ translates into the formula:
$$ \texttt{Variables}[0, x] = x \quad \mbox{for all $x \in \{0, \cdots, n-1\}$.} $$
The function `leftNeutral` takes two arguments:
* `n` is the size of the group.
* `Variables` is the map that encodes the binary operation of the group.

The function returns a list of constraints specifying that $0$ is the *left neutral element* of the group.

In [6]:
let leftNeutral (n: int) (Variables: Map<int * int, IntExpr>) : BoolExpr list =
    [ for x in 0 .. n - 1 -> Variables[(0, x)] .= x ]

In [7]:
leftNeutral 3 Variables

[V00 = 0; V01 = 1; V02 = 2]

The axiom
$$ \forall x: \exists y: y \circ x = \mathrm{e} $$
that every element $x$ has a left inverse element $y$ translates into the formula
$$ \forall x \in \{0, \cdots, n-1\}: \exists y \in \{0, \cdots, n-1\}: \texttt{Variables}[y, x] = 0. $$
The function `leftInverse` takes two arguments:
* `n` is the size of the group.
* `Variables` is the map that encodes the binary operation of the group.

The function returns a list of constraints that specify that every element has a left inverse element.
Since the neutral element $0$ is already its own left inverse, for reasons of efficiency this element is excluded.
Furthermore, we already know that the neutral element $0$ can never be the left inverse element of an
element that is different from $0$.

In [8]:
let leftInverse (n: int) (Variables: Map<int * int, IntExpr>) : BoolExpr list =
    [ for x in 1 .. n - 1 -> Or [ for y in 1 .. n - 1 -> Variables[(y, x)] .= 0 ] ]

In [9]:
leftInverse 3 Variables

[V11 = 0 ∨ V21 = 0; V12 = 0 ∨ V22 = 0]

The axiom
$$ \forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z) $$
translates into the formula
$$ \forall x \in G: 
   \forall y \in G:
   \forall z \in G:
   \forall u \in G:
   \forall v \in G:
     \bigl(\texttt{Variables}[x, y] = u \wedge \texttt{Variables}[y, z] = v \rightarrow \texttt{Variables}[u, z] = \texttt{Variables}[x, v]\bigr), $$
where we have defined $G = \{0, \cdots n-1\}$.
The function `associative` takes two arguments:
* `n` is the size of the group.
* `Variables` is the map that encodes the binary operation of the group.

The function returns a list of constraints that specify that the binary operation is associative.

In [10]:
let associative (n: int) (Variables: Map<int * int, IntExpr>) : BoolExpr list =
    let L = [ 0 .. n - 1 ]
    [ for x in L do
        for y in L do
          for z in L do
            for xy in L do
              for yz in L ->
                Implies(And [ Variables[(x, y)] .= xy; Variables[(y, z)] .= yz ],
                        Variables[(xy, z)] .= Variables[(x, yz)]) ]

In [11]:
(associative 3 Variables).Length

243

The function `nonCommute` takes two arguments:
* `n` is the size of the group.
* `Variables` is the map that encodes the binary operation of the group.

The function returns a list containing one constraint that specifies that the binary operation $\circ$ is 
not commutative, i.e. there is some $x$ and some $y$ such that $x \circ y \not= y \circ x$.

In [12]:
let nonCommute (n: int) (Variables: Map<int * int, IntExpr>) : BoolExpr list =
    [ Or [ for x in 1 .. n - 1 do
             for y in 1 .. n - 1 do
               if x <> y then Variables[(x, y)] .<> Variables[(y, x)] ] ]

In [13]:
nonCommute 3 Variables

[¬(V12 = V21) ∨ ¬(V21 = V12)]

All group elements are from the set $\{0,1,\cdots,n-1\}$.
The function `inRange` takes two arguments:
* `n` is the size of the group.
* `Variables` is the map that encodes the binary operation of the group.

The function returns a list of constraints that specify that the values of all variables are elements of
the set $\{0,1,\cdots,n-1\}$.

In [14]:
let inRange (n: int) (Variables: Map<int * int, IntExpr>) : BoolExpr list =
    [ for x in 0 .. n - 1 do
        for y in 0 .. n - 1 do
          yield 0 .<= Variables[(x, y)]
          yield Variables[(x, y)] .< n ]

The function `allConstraints` takes the same two arguments.  It returns the list of all constraints that specify a non-commutative group with $n$ elements.

In [15]:
let allConstraints (n: int) (Variables: Map<int * int, IntExpr>) : BoolExpr list =
    leftNeutral n Variables
    @ leftInverse n Variables
    @ associative n Variables
    @ nonCommute  n Variables
    @ inRange     n Variables

In [16]:
(allConstraints 3 Variables).Length

267

The function `solve` takes one argument:
- `n` is the size of the group that should be constructed.

The function tries to construct a non-commutative group of size `n`.  It returns `Some Solution` if such a group exists, where `Solution` maps the pairs $(x, y)$ to the products $x \circ y$.  Otherwise, `None` is returned.  Note that this function hides the function `solve` from `z3.fsx`.

In [17]:
let solve (n: int) : Map<int * int, int> option =
    let Variables = Map [ for row in 0 .. n - 1 do
                            for col in 0 .. n - 1 ->
                                ((row, col), Int (sprintf "V%d%d" row col)) ]
    let S = newSolver ()
    let Constraints = allConstraints n Variables
    printfn "Number of constraints: %d." Constraints.Length
    S.Add(Array.ofList Constraints)
    if S.Check() = Status.SATISFIABLE then
        let M = S.Model
        Some (Map [ for KeyValue(key, v) in Variables -> (key, intValue M v) ])
    else
        None

The function `findNonCommutativeGroup` takes no arguments.  It tries $n = 1, 2, 3, \cdots$ until a non-commutative group with $n$ elements is found.

In [18]:
let findNonCommutativeGroup () : Map<int * int, int> =
    let rec search (n: int) =
        printfn "Testing n = %d ..." n
        match solve n with
        | Some solution -> solution
        | None          -> search (n + 1)
    search 1

In [19]:
#!time
let Solution = findNonCommutativeGroup ()
Solution

Testing n = 

1

 ...

Number of constraints: 

5

...

Number of constraints: 

7860

.

map [((0, 0), 0); ((0, 1), 1); ((0, 2), 2); ((0, 3), 3); ((0, 4), 4); ((0, 5), 5); ((1, 0), 1); ((1, 1), 0); ((1, 2), 3); ((1, 3), 2); ((1, 4), 5); ((1, 5), 4); ((2, 0), 2); ((2, 1), 4); ((2, 2), 5); ((2, 3), 1); ((2, 4), 3); ((2, 5), 0); ((3, 0), 3); ((3, 1), 5); ((3, 2), 4); ((3, 3), 0); ((3, 4), 2); ((3, 5), 1); ((4, 0), 4); ((4, 1), 2); ((4, 2), 1); ((4, 3), 5); ((4, 4), 0); ((4, 5), 3); ((5, 0), 5); ((5, 1), 3); ((5, 2), 0); ((5, 3), 4); ((5, 4), 1); ((5, 5), 2)]

Wall time: 249.7484ms

## Graphical Representation

The function `show` takes one argument:
- `Solution` maps pairs $(x, y)$ to the products $x \circ y$.

It prints the multiplication table of the group.  The row index is $x$, the column index is $y$.  Note that this function hides the function `show` from `setup.fsx`.

In [20]:
let show (Solution: Map<int * int, int>) : unit =
    let n = int (round (sqrt (float Solution.Count)))
    let line (left: string) (middle: string) (right: string) =
        left + String.concat middle (List.replicate n "═══") + right
    printfn "%s" (line "╔" "╦" "╗")
    for row in 0 .. n - 1 do
        printfn "║%s" (String.concat "" [ for col in 0 .. n - 1 -> sprintf " %d ║" Solution[(row, col)] ])
        if row < n - 1 then
            printfn "%s" (line "╠" "╬" "╣")
    printfn "%s" (line "╚" "╩" "╝")

In [21]:
show Solution

╔═══╦═══╦═══╦═══╦═══╦═══╗

║

 0 ║ 1 ║ 2 ║ 3 ║ 4 ║ 5 ║

╠═══╬═══╬═══╬═══╬═══╬═══╣

║

...

 4 ║ 2 ║ 1 ║ 5 ║ 0 ║ 3 ║

╠═══╬═══╬═══╬═══╬═══╬═══╣

║

 5 ║ 3 ║ 0 ║ 4 ║ 1 ║ 2 ║

╚═══╩═══╩═══╩═══╩═══╩═══╝